# Notebook 09: independent three-hour deadline watcher

Tagged definitions only. The watcher preserves the original clock and can terminate only this study's named tmux process tree. It retains valid atomic checkpoints and marks a cutoff INCOMPLETE. The module itself launches no worker.

In [ ]:
"""Separate process enforces the first user launch's absolute wall-clock limit."""
import json, os, shlex, signal, subprocess, sys, time
from pathlib import Path
import nbformat

def _atomic(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2) + "\n")
    temp.replace(path)

def _exists(session):
    return subprocess.run(["tmux","has-session","-t","="+session],capture_output=True).returncode == 0

def _terminate_session_tree(session):
    if not _exists(session):
        return []
    roots = {int(x) for x in subprocess.check_output(["tmux","list-panes","-t","="+session,"-F","#{pane_pid}"],text=True).split()}
    edges = [tuple(map(int, line.split())) for line in subprocess.check_output(["ps","-eo","pid=,ppid="],text=True).splitlines()]
    owned = set(roots)
    while True:
        expanded = owned | {pid for pid,parent in edges if parent in owned}
        if expanded == owned:
            break
        owned = expanded
    for pid in sorted(owned-roots,reverse=True) + sorted(roots):
        try:
            os.kill(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    subprocess.run(["tmux","kill-session","-t","="+session],capture_output=True)
    return sorted(owned)

def run_watchdog(artifact, session, poll_seconds=10):
    artifact = Path(artifact)
    clock = json.loads((artifact/"budget_clock.json").read_text())
    binding = json.loads((artifact/"study_binding.json").read_text())
    if binding["budget_clock"] != clock:
        raise ValueError("Study clock changed from its original registered value")
    deadline = float(clock["deadline_unix"])
    while True:
        path = artifact/"study_status.json"
        status = json.loads(path.read_text()) if path.exists() else {}
        if status.get("status") == "COMPLETED" or (status.get("status") in ("FAILED","INCOMPLETE") and not _exists(session)):
            result = {"status":"EXITED_TERMINAL_STATE","worker_status":status["status"],
                      "observed_unix":time.time(),"deadline_unix":deadline}
            _atomic(artifact/"deadline_watchdog_result.json",result)
            return result
        remaining = deadline-time.time()
        if remaining <= 0:
            killed = _terminate_session_tree(session)
            result = {"status":"DEADLINE_ENFORCED","deadline_unix":deadline,"enforced_unix":time.time(),"terminated_pids":killed}
            _atomic(path,{"stage":"absolute_deadline","status":"INCOMPLETE","updated_unix":time.time(),
                "reason":"The independent watcher enforced the original configured time limit. Latest valid atomic checkpoints are retained; no clock extension.",
                "elapsed_seconds":time.time()-clock["started_unix"]})
            _atomic(artifact/"deadline_watchdog_result.json",result)
            return result
        time.sleep(min(remaining,max(.01,float(poll_seconds))))

def launch_watchdog(root, run_id):
    root = Path(root)
    artifact = root/"artifacts/notebook_09"/run_id
    session, worker = "fmm-deep-energy-deadline-"+run_id, "fmm-deep-energy-"+run_id
    if _exists(session):
        return {"status":"EXISTING_SESSION","session":session}
    status_path = artifact/"study_status.json"
    if status_path.exists() and json.loads(status_path.read_text()).get("status") == "COMPLETED":
        return {"status":"COMPLETED"}
    module = root/"experiments/09_deep_energy/deadline_watchdog.ipynb"
    code = "from pathlib import Path\nimport json\nns={}\np=Path("+repr(str(module))+")\nfor c in json.loads(p.read_text())['cells']:\n    if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[]):\n        exec(compile(''.join(c['source']),str(p)+'#ahr-export','exec'),ns)\nns['run_watchdog'](Path("+repr(str(artifact))+"),"+repr(worker)+")\n"
    notebook = artifact/"deadline_watchdog.ipynb"
    nbformat.write(nbformat.v4.new_notebook(cells=[
        nbformat.v4.new_markdown_cell("# Notebook09 independent deadline watcher\nOnly this study's explicitly named worker process tree can be terminated. Atomic checkpoints and the original clock remain intact."),
        nbformat.v4.new_code_cell(code)],metadata={"kernelspec":{"display_name":"Python (ahr-go)","language":"python","name":"ahr-go"}}),notebook)
    command = [sys.executable,"-m","jupyter","nbconvert","--execute","--to","notebook","--inplace",
               "--ExecutePreprocessor.timeout=-1","--ExecutePreprocessor.kernel_name=ahr-go",str(notebook)]
    shell = "exec "+shlex.join(command)+" > "+shlex.quote(str(artifact/"deadline_watchdog.log"))+" 2>&1"
    subprocess.run(["tmux","new-session","-d","-s",session,"-c",str(root),"bash","-c",shell],check=True)
    result = {"status":"LAUNCHED","session":session,"worker_session":worker,
              "deadline_unix":json.loads((artifact/"budget_clock.json").read_text())["deadline_unix"]}
    _atomic(artifact/"deadline_watchdog_launch.json",result)
    return result
